# Driver Drowsiness Detection - Model Training (Google Colab)

This notebook trains a CNN (Convolutional Neural Network) to classify eye images as **Drowsy** (closed) or **Non_Drowsy** (open), then downloads the trained model for use in the web app.

## How to use this notebook
1. Runtime menu -> Change runtime type -> select **T4 GPU** -> Save (makes training much faster)
2. Run each cell below, top to bottom (Shift+Enter)
3. When prompted, upload your `dataset.zip` file (see step below for how to prepare it)
4. At the end, two files will auto-download: `drowsiness_model.h5` and `class_indices.json`
5. Put both files into your project folder (next to `app.py`)

## Preparing dataset.zip (do this on your own computer first)
Your `dataset` folder should look like this:
```
dataset/
    Drowsy/        <- closed-eye images
    Non_Drowsy/    <- open-eye images
```
Right-click the `dataset` folder -> **Send to -> Compressed (zipped) folder** (Windows) to create `dataset.zip`, then upload that zip in the cell below.


In [ ]:
# STEP 1: Upload your dataset.zip
from google.colab import files
uploaded = files.upload()   # select your dataset.zip when prompted


In [ ]:
# STEP 2: Unzip it
import zipfile, os

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('.')

# Find the dataset folder (handles case where zip contains a nested folder)
for root, dirs, files_ in os.walk('.'):
    if 'Drowsy' in dirs and 'Non_Drowsy' in dirs:
        DATASET_DIR = root
        break

print("Dataset found at:", DATASET_DIR)
print("Drowsy images:", len(os.listdir(os.path.join(DATASET_DIR, 'Drowsy'))))
print("Non_Drowsy images:", len(os.listdir(os.path.join(DATASET_DIR, 'Non_Drowsy'))))


## Step 3: Preprocessing + Data Augmentation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

IMG_SIZE = 96
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.15,
    validation_split=0.30
)

train_data = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='training'
)
val_data = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation'
)

print("Class indices:", train_data.class_indices)


## Step 4: Build the CNN model (Artificial Neural Network)

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


## Step 5: Train the model\n\nOn Colab's free GPU this should take just a couple of minutes.

In [ ]:
EPOCHS = 15
history = model.fit(train_data, validation_data=val_data, epochs=EPOCHS)


In [ ]:
# Plot accuracy/loss
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend(); plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend(); plt.title('Loss')
plt.savefig('training_graph.png')
plt.show()


## Step 6: Evaluate properly (correct label order -- shuffle disabled)

In [ ]:
eval_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.30)
eval_data = eval_datagen.flow_from_directory(
    DATASET_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation', shuffle=False
)

y_true = eval_data.classes
y_pred_prob = model.predict(eval_data)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=list(eval_data.class_indices.keys())))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(eval_data.class_indices.keys()),
            yticklabels=list(eval_data.class_indices.keys()))
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png')
plt.show()


## Step 7: Save the model + label mapping, then download both\n\nThese two files are what you need for the web app.

In [ ]:
model.save('drowsiness_model.h5')

with open('class_indices.json', 'w') as f:
    json.dump(train_data.class_indices, f)

print("Saved drowsiness_model.h5 and class_indices.json")


In [ ]:
from google.colab import files as colab_files
colab_files.download('drowsiness_model.h5')
colab_files.download('class_indices.json')
print("Downloads triggered -- check your browser's download folder.")
print("Also download training_graph.png and confusion_matrix.png from the Colab file browser (left sidebar) for your report.")
